In [2]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached scikit_learn-1.9.1-cp311-cp311-win_amd64.whl (8.3 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/

In [3]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Libraries imported successfully!")


Libraries imported successfully!


In [4]:

# Project directories

PROJECT_DIR = Path(
    r"D:\AtmoGraph-Supply-Chain-Ripple-Predictor"
)

DATA_DIR = Path(
    r"D:\AtmoGraph-Supply-Chain-Ripple-Predictor-Krish\graph_database\data"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "processed_features"

NODE_OUTPUT_DIR = OUTPUT_DIR / "nodes"
EDGE_OUTPUT_DIR = OUTPUT_DIR / "edges"
METADATA_DIR = OUTPUT_DIR / "metadata"

# Create output directories
for folder in [NODE_OUTPUT_DIR, EDGE_OUTPUT_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project paths configured successfully!")
print("Data directory exists:", DATA_DIR.exists())
print("Output directory:", OUTPUT_DIR)


Project paths configured successfully!
Data directory exists: True
Output directory: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\outputs\processed_features


In [5]:

# Load node datasets

nodes = {
    "suppliers": pd.read_csv(DATA_DIR / "nodes_suppliers.csv"),
    "manufacturers": pd.read_csv(DATA_DIR / "nodes_manufacturers.csv"),
    "ports": pd.read_csv(DATA_DIR / "nodes_ports.csv"),
    "distributors": pd.read_csv(DATA_DIR / "nodes_distributors.csv"),
    "disruption_events": pd.read_csv(DATA_DIR / "nodes_disruption_events.csv")
}

# Load relationship datasets

edges = {
    "supplies_to": pd.read_csv(DATA_DIR / "edges_supplies_to.csv"),
    "ships_via": pd.read_csv(DATA_DIR / "edges_ships_via.csv"),
    "route_to": pd.read_csv(DATA_DIR / "edges_route_to.csv"),
    "delivers_to": pd.read_csv(DATA_DIR / "edges_delivers_to.csv"),
    "had_disruption": pd.read_csv(DATA_DIR / "edges_had_disruption.csv")
}

print("Updated graph datasets loaded successfully!")


Updated graph datasets loaded successfully!


In [6]:

# Cell 4: Verify Loaded Datasets

print("NODE DATASETS")
print("=" * 50)

total_nodes = 0

for name, df in nodes.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
    total_nodes += len(df)

print("\nTotal nodes:", total_nodes)

print("\nRELATIONSHIP DATASETS")
print("=" * 50)

total_edges = 0

for name, df in edges.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
    total_edges += len(df)

print("\nTotal relationships:", total_edges)

assert total_nodes == 45, "Unexpected node count!"
assert total_edges == 108, "Unexpected relationship count!"

print("\nDataset verification successful!")


NODE DATASETS
suppliers: 5 rows, 13 columns
manufacturers: 5 rows, 13 columns
ports: 13 rows, 12 columns
distributors: 10 rows, 8 columns
disruption_events: 12 rows, 10 columns

Total nodes: 45

RELATIONSHIP DATASETS
supplies_to: 25 rows, 8 columns
ships_via: 9 rows, 9 columns
route_to: 50 rows, 15 columns
delivers_to: 12 rows, 7 columns
had_disruption: 12 rows, 3 columns

Total relationships: 108

Dataset verification successful!


In [7]:

# Cell 5: Define Candidate Node Features

node_feature_candidates = {
    "suppliers": {
        "numerical": [
            "lat",
            "lon",
            "sku_count",
            "avg_defect_rate",
            "avg_supplier_lead_time_days",
            "inspection_fail_share"
        ],
        "categorical": [
            "primary_city"
        ]
    },

    "manufacturers": {
        "numerical": [
            "lat",
            "lon",
            "sku_count",
            "production_volume",
            "avg_manufacturing_lead_time_days",
            "avg_manufacturing_cost",
            "avg_defect_rate"
        ],
        "categorical": [
            "city"
        ]
    },

    "ports": {
        "numerical": [
            "lat",
            "lon",
            "hist_disruption_events",
            "hist_avg_affected_days",
            "hist_max_severity"
        ],
        "categorical": [
            "country"
        ]
    },

    "distributors": {
        "numerical": [
            "lat",
            "lon"
        ],
        "categorical": [
            "country"
        ]
    },

    "disruption_events": {
        "numerical": [
            "year"
        ],
        "categorical": []
    }
}

print("Candidate node features defined successfully!")


Candidate node features defined successfully!


In [8]:

# Cell 6: Validate Candidate Node Features

for node_type, feature_groups in node_feature_candidates.items():

    df = nodes[node_type]

    selected_columns = (
        feature_groups["numerical"]
        + feature_groups["categorical"]
    )

    print(f"\n{'=' * 55}")
    print(f"NODE TYPE: {node_type.upper()}")
    print("=" * 55)

    missing_columns = [
        col for col in selected_columns
        if col not in df.columns
    ]

    if missing_columns:
        print("Missing columns:", missing_columns)
    else:
        print("All selected columns are available.")

    print("\nMissing values:")
    display(df[selected_columns].isnull().sum())

    print("\nSelected feature count:", len(selected_columns))



NODE TYPE: SUPPLIERS
All selected columns are available.

Missing values:


lat                            0
lon                            0
sku_count                      0
avg_defect_rate                0
avg_supplier_lead_time_days    0
inspection_fail_share          0
primary_city                   0
dtype: int64


Selected feature count: 7

NODE TYPE: MANUFACTURERS
All selected columns are available.

Missing values:


lat                                 0
lon                                 0
sku_count                           0
production_volume                   0
avg_manufacturing_lead_time_days    0
avg_manufacturing_cost              0
avg_defect_rate                     0
city                                0
dtype: int64


Selected feature count: 8

NODE TYPE: PORTS
All selected columns are available.

Missing values:


lat                       0
lon                       0
hist_disruption_events    0
hist_avg_affected_days    9
hist_max_severity         9
country                   0
dtype: int64


Selected feature count: 6

NODE TYPE: DISTRIBUTORS
All selected columns are available.

Missing values:


lat        0
lon        0
country    0
dtype: int64


Selected feature count: 3

NODE TYPE: DISRUPTION_EVENTS
All selected columns are available.

Missing values:


year    0
dtype: int64


Selected feature count: 1


In [14]:

edge_feature_candidates = {
    "supplies_to": {
        "numerical": [
            "lead_time_days",
            "volume",
            "dependency_weight",
            "n_skus"
        ],
        "categorical": ["transport_mode"]
    },

    "ships_via": {
        "numerical": [
            "lead_time_days",
            "volume",
            "dependency_weight",
            "distance_km"
        ],
        "categorical": []
    },

    "route_to": {
        "numerical": [
            "lead_time_days",
            "volume",
            "distance_km",
            "avg_freight_cost_usd",
            "avg_port_congestion",
            "dependency_weight"
        ],
        "categorical": [
            "transport_mode",
            "cargo_type"
        ]
    },

   "delivers_to": {
    "numerical": [
        "lead_time_days",
        "volume",
        "dependency_weight"
    ],
    "categorical": []
},

    "had_disruption": {
        "numerical": [],
        "categorical": []
    }
}

print("Revised relationship feature candidates defined successfully!")


Revised relationship feature candidates defined successfully!


In [15]:

for edge_type, feature_groups in edge_feature_candidates.items():
    df = edges[edge_type]

    selected_columns = (
        feature_groups["numerical"] +
        feature_groups["categorical"]
    )

    print(f"\n{'=' * 55}")
    print(f"RELATIONSHIP TYPE: {edge_type.upper()}")
    print("=" * 55)

    missing_columns = [
        col for col in selected_columns
        if col not in df.columns
    ]

    if missing_columns:
        print("Missing columns:", missing_columns)
    elif selected_columns:
        print("All selected columns are available.")
        print("\nMissing values:")
        display(df[selected_columns].isnull().sum())
    else:
        print("No additional relationship features selected.")

    print("\nSelected feature count:", len(selected_columns))



RELATIONSHIP TYPE: SUPPLIES_TO
All selected columns are available.

Missing values:


lead_time_days       0
volume               0
dependency_weight    0
n_skus               0
transport_mode       0
dtype: int64


Selected feature count: 5

RELATIONSHIP TYPE: SHIPS_VIA
All selected columns are available.

Missing values:


lead_time_days       0
volume               0
dependency_weight    0
distance_km          0
dtype: int64


Selected feature count: 4

RELATIONSHIP TYPE: ROUTE_TO
All selected columns are available.

Missing values:


lead_time_days          0
volume                  0
distance_km             0
avg_freight_cost_usd    0
avg_port_congestion     0
dependency_weight       0
transport_mode          0
cargo_type              0
dtype: int64


Selected feature count: 8

RELATIONSHIP TYPE: DELIVERS_TO
All selected columns are available.

Missing values:


lead_time_days       0
volume               0
dependency_weight    0
dtype: int64


Selected feature count: 3

RELATIONSHIP TYPE: HAD_DISRUPTION
No additional relationship features selected.

Selected feature count: 0


In [16]:

for edge_type, feature_groups in edge_feature_candidates.items():

    df = edges[edge_type]

    print(f"\n{'=' * 60}")
    print(f"RELATIONSHIP: {edge_type.upper()}")
    print("=" * 60)

    for col in feature_groups["numerical"]:
        print(f"\n{col}")
        print("Unique values:", df[col].nunique())
        print(df[col].describe())

    for col in feature_groups["categorical"]:
        print(f"\n{col}")
        print(df[col].value_counts())



RELATIONSHIP: SUPPLIES_TO

lead_time_days
Unique values: 24
count    25.000000
mean     17.632000
std       6.218341
min       3.000000
25%      15.250000
50%      17.500000
75%      22.330000
max      28.000000
Name: lead_time_days, dtype: float64

volume
Unique values: 25
count     25.000000
mean     196.880000
std       94.956709
min        9.000000
25%      140.000000
50%      194.000000
75%      270.000000
max      406.000000
Name: volume, dtype: float64

dependency_weight
Unique values: 25
count    25.000000
mean      0.200000
std       0.098201
min       0.012300
25%       0.117300
50%       0.199400
75%       0.248000
max       0.431100
Name: dependency_weight, dtype: float64

n_skus
Unique values: 7
count    25.000000
mean      4.000000
std       1.581139
min       1.000000
25%       3.000000
50%       4.000000
75%       5.000000
max       8.000000
Name: n_skus, dtype: float64

transport_mode
transport_mode
Air     8
Rail    8
Road    7
Sea     2
Name: count, dtype: int64

RE

In [17]:

print("CONSTANT FEATURE CHECK")
print("=" * 65)

for edge_type, feature_groups in edge_feature_candidates.items():

    df = edges[edge_type]

    print(f"\n{edge_type.upper()}")
    print("-" * 50)

    selected_columns = (
        feature_groups["numerical"] +
        feature_groups["categorical"]
    )

    if not selected_columns:
        print("No additional features.")
        continue

    for col in selected_columns:
        unique_count = df[col].nunique()

        status = "CONSTANT" if unique_count <= 1 else "VARIABLE"

        print(f"{col}: {unique_count} unique values ({status})")


CONSTANT FEATURE CHECK

SUPPLIES_TO
--------------------------------------------------
lead_time_days: 24 unique values (VARIABLE)
volume: 25 unique values (VARIABLE)
dependency_weight: 25 unique values (VARIABLE)
n_skus: 7 unique values (VARIABLE)
transport_mode: 4 unique values (VARIABLE)

SHIPS_VIA
--------------------------------------------------
lead_time_days: 5 unique values (VARIABLE)
volume: 9 unique values (VARIABLE)
dependency_weight: 7 unique values (VARIABLE)
distance_km: 9 unique values (VARIABLE)

ROUTE_TO
--------------------------------------------------
lead_time_days: 31 unique values (VARIABLE)
volume: 50 unique values (VARIABLE)
distance_km: 50 unique values (VARIABLE)
avg_freight_cost_usd: 50 unique values (VARIABLE)
avg_port_congestion: 50 unique values (VARIABLE)
dependency_weight: 48 unique values (VARIABLE)
transport_mode: 4 unique values (VARIABLE)
cargo_type: 5 unique values (VARIABLE)

DELIVERS_TO
--------------------------------------------------
lead_tim

In [18]:

node_feature_frames = {}

for node_type, feature_groups in node_feature_candidates.items():

    df = nodes[node_type].copy()

    numerical_cols = feature_groups["numerical"]
    categorical_cols = feature_groups["categorical"]

    selected_cols = numerical_cols + categorical_cols

    # Create a separate feature dataframe
    feature_df = df[selected_cols].copy()

    # Handle missing numerical values
    for col in numerical_cols:

        if feature_df[col].isnull().any():

            # Preserve missingness information
            feature_df[f"{col}_missing"] = (
                feature_df[col].isnull().astype(int)
            )

            # Replace missing values with the median
            median_value = feature_df[col].median()

            feature_df[col] = feature_df[col].fillna(
                median_value
            )

    node_feature_frames[node_type] = feature_df

    print(f"\n{node_type.upper()}")
    print("-" * 45)
    print("Feature shape:", feature_df.shape)
    print("Remaining missing values:", feature_df.isnull().sum().sum())

print("\nNode feature datasets created successfully!")



SUPPLIERS
---------------------------------------------
Feature shape: (5, 7)
Remaining missing values: 0

MANUFACTURERS
---------------------------------------------
Feature shape: (5, 8)
Remaining missing values: 0

PORTS
---------------------------------------------
Feature shape: (13, 8)
Remaining missing values: 0

DISTRIBUTORS
---------------------------------------------
Feature shape: (10, 3)
Remaining missing values: 0

DISRUPTION_EVENTS
---------------------------------------------
Feature shape: (12, 1)
Remaining missing values: 0

Node feature datasets created successfully!


In [19]:

from sklearn.preprocessing import OneHotEncoder

encoded_node_features = {}
node_encoders = {}

for node_type, feature_groups in node_feature_candidates.items():

    feature_df = node_feature_frames[node_type].copy()
    categorical_cols = feature_groups["categorical"]

    if categorical_cols:

        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

        encoded_array = encoder.fit_transform(
            feature_df[categorical_cols]
        )

        encoded_columns = encoder.get_feature_names_out(
            categorical_cols
        )

        encoded_df = pd.DataFrame(
            encoded_array,
            columns=encoded_columns,
            index=feature_df.index
        )

        feature_df = feature_df.drop(
            columns=categorical_cols
        )

        feature_df = pd.concat(
            [feature_df, encoded_df],
            axis=1
        )

        node_encoders[node_type] = encoder

    encoded_node_features[node_type] = feature_df

    print(f"\n{node_type.upper()}")
    print("-" * 45)
    print("Encoded feature shape:", feature_df.shape)
    print("Remaining categorical columns:",
          list(feature_df.select_dtypes(include=["object"]).columns))

print("\nCategorical encoding completed successfully!")



SUPPLIERS
---------------------------------------------
Encoded feature shape: (5, 9)
Remaining categorical columns: []

MANUFACTURERS
---------------------------------------------
Encoded feature shape: (5, 12)
Remaining categorical columns: []

PORTS
---------------------------------------------
Encoded feature shape: (13, 17)
Remaining categorical columns: []

DISTRIBUTORS
---------------------------------------------
Encoded feature shape: (10, 12)
Remaining categorical columns: []

DISRUPTION_EVENTS
---------------------------------------------
Encoded feature shape: (12, 1)
Remaining categorical columns: []

Categorical encoding completed successfully!


In [20]:

from sklearn.preprocessing import StandardScaler

scaled_node_features = {}
node_scalers = {}

for node_type, feature_groups in node_feature_candidates.items():

    feature_df = encoded_node_features[node_type].copy()

    # Original numerical columns
    numerical_cols = feature_groups["numerical"].copy()

    # Include missingness indicators as binary numerical features
    missing_cols = [
        col for col in feature_df.columns
        if col.endswith("_missing")
    ]

    scaling_cols = [
        col for col in numerical_cols
        if col in feature_df.columns
    ]

    # Scale continuous numerical features, not binary indicators
    scaler = StandardScaler()

    if scaling_cols:
        feature_df[scaling_cols] = scaler.fit_transform(
            feature_df[scaling_cols]
        )

        node_scalers[node_type] = scaler

    scaled_node_features[node_type] = feature_df

    print(f"\n{node_type.upper()}")
    print("-" * 45)
    print("Feature shape:", feature_df.shape)
    print("Scaled columns:", scaling_cols)
    print("Missingness indicators preserved:", missing_cols)

print("\nNode numerical scaling completed successfully!")



SUPPLIERS
---------------------------------------------
Feature shape: (5, 9)
Scaled columns: ['lat', 'lon', 'sku_count', 'avg_defect_rate', 'avg_supplier_lead_time_days', 'inspection_fail_share']
Missingness indicators preserved: []

MANUFACTURERS
---------------------------------------------
Feature shape: (5, 12)
Scaled columns: ['lat', 'lon', 'sku_count', 'production_volume', 'avg_manufacturing_lead_time_days', 'avg_manufacturing_cost', 'avg_defect_rate']
Missingness indicators preserved: []

PORTS
---------------------------------------------
Feature shape: (13, 17)
Scaled columns: ['lat', 'lon', 'hist_disruption_events', 'hist_avg_affected_days', 'hist_max_severity']
Missingness indicators preserved: ['hist_avg_affected_days_missing', 'hist_max_severity_missing']

DISTRIBUTORS
---------------------------------------------
Feature shape: (10, 12)
Scaled columns: ['lat', 'lon']
Missingness indicators preserved: []

DISRUPTION_EVENTS
---------------------------------------------
Fe

In [21]:

print("PROCESSED NODE FEATURE VALIDATION")
print("=" * 65)

for node_type, feature_df in scaled_node_features.items():

    original_count = len(nodes[node_type])
    processed_count = len(feature_df)

    print(f"\n{node_type.upper()}")
    print("-" * 50)

    print("Original nodes:", original_count)
    print("Processed rows:", processed_count)
    print("Feature columns:", feature_df.shape[1])

    print(
        "All numerical:",
        all(pd.api.types.is_numeric_dtype(dtype)
            for dtype in feature_df.dtypes)
    )

    print("Missing values:", feature_df.isnull().sum().sum())

    print(
        "Infinite values:",
        np.isinf(feature_df.to_numpy(dtype=float)).sum()
    )

    assert original_count == processed_count
    assert feature_df.isnull().sum().sum() == 0
    assert np.isfinite(feature_df.to_numpy(dtype=float)).all()

print("\nAll node feature validation checks passed!")


PROCESSED NODE FEATURE VALIDATION

SUPPLIERS
--------------------------------------------------
Original nodes: 5
Processed rows: 5
Feature columns: 9
All numerical: True
Missing values: 0
Infinite values: 0

MANUFACTURERS
--------------------------------------------------
Original nodes: 5
Processed rows: 5
Feature columns: 12
All numerical: True
Missing values: 0
Infinite values: 0

PORTS
--------------------------------------------------
Original nodes: 13
Processed rows: 13
Feature columns: 17
All numerical: True
Missing values: 0
Infinite values: 0

DISTRIBUTORS
--------------------------------------------------
Original nodes: 10
Processed rows: 10
Feature columns: 12
All numerical: True
Missing values: 0
Infinite values: 0

DISRUPTION_EVENTS
--------------------------------------------------
Original nodes: 12
Processed rows: 12
Feature columns: 1
All numerical: True
Missing values: 0
Infinite values: 0

All node feature validation checks passed!


In [22]:
#Create relationship feature datasets

edge_feature_frames = {}
edge_index_frames = {}

for edge_type, feature_groups in edge_feature_candidates.items():

    df = edges[edge_type].copy()

    # Preserve graph connectivity separately
    edge_index_frames[edge_type] = df[["start_id", "end_id"]].copy()

    # Select only model features
    selected_cols = (
        feature_groups["numerical"] +
        feature_groups["categorical"]
    )

    feature_df = df[selected_cols].copy()

    edge_feature_frames[edge_type] = feature_df

    print(f"\n{edge_type.upper()}")
    print("-" * 45)
    print("Relationship count:", len(df))
    print("Feature shape:", feature_df.shape)
    print("Connectivity shape:", edge_index_frames[edge_type].shape)
    print("Missing values:", feature_df.isnull().sum().sum())

print("\nRelationship feature datasets created successfully!")



SUPPLIES_TO
---------------------------------------------
Relationship count: 25
Feature shape: (25, 5)
Connectivity shape: (25, 2)
Missing values: 0

SHIPS_VIA
---------------------------------------------
Relationship count: 9
Feature shape: (9, 4)
Connectivity shape: (9, 2)
Missing values: 0

ROUTE_TO
---------------------------------------------
Relationship count: 50
Feature shape: (50, 8)
Connectivity shape: (50, 2)
Missing values: 0

DELIVERS_TO
---------------------------------------------
Relationship count: 12
Feature shape: (12, 3)
Connectivity shape: (12, 2)
Missing values: 0

HAD_DISRUPTION
---------------------------------------------
Relationship count: 12
Feature shape: (12, 0)
Connectivity shape: (12, 2)
Missing values: 0.0

Relationship feature datasets created successfully!


In [23]:

# Encode categorical features for each relationship type using One-Hot Encoding.

from sklearn.preprocessing import OneHotEncoder

encoded_edge_features = {}
edge_encoders = {}

for edge_type, feature_groups in edge_feature_candidates.items():

    feature_df = edge_feature_frames[edge_type].copy()
    categorical_cols = feature_groups["categorical"]

    if categorical_cols:

        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

        encoded_array = encoder.fit_transform(
            feature_df[categorical_cols]
        )

        encoded_columns = encoder.get_feature_names_out(
            categorical_cols
        )

        encoded_df = pd.DataFrame(
            encoded_array,
            columns=encoded_columns,
            index=feature_df.index
        )

        feature_df = feature_df.drop(
            columns=categorical_cols
        )

        feature_df = pd.concat(
            [feature_df, encoded_df],
            axis=1
        )

        edge_encoders[edge_type] = encoder

    encoded_edge_features[edge_type] = feature_df

    print(f"\n{edge_type.upper()}")
    print("-" * 45)
    print("Encoded feature shape:", feature_df.shape)
    print(
        "Remaining categorical columns:",
        list(feature_df.select_dtypes(include=["object"]).columns)
    )

print("\nRelationship categorical encoding completed successfully!")



SUPPLIES_TO
---------------------------------------------
Encoded feature shape: (25, 8)
Remaining categorical columns: []

SHIPS_VIA
---------------------------------------------
Encoded feature shape: (9, 4)
Remaining categorical columns: []

ROUTE_TO
---------------------------------------------
Encoded feature shape: (50, 15)
Remaining categorical columns: []

DELIVERS_TO
---------------------------------------------
Encoded feature shape: (12, 3)
Remaining categorical columns: []

HAD_DISRUPTION
---------------------------------------------
Encoded feature shape: (12, 0)
Remaining categorical columns: []

Relationship categorical encoding completed successfully!


In [24]:

# Standardize numerical features separately for each relationship type.

from sklearn.preprocessing import StandardScaler

scaled_edge_features = {}
edge_scalers = {}

for edge_type, feature_groups in edge_feature_candidates.items():

    feature_df = encoded_edge_features[edge_type].copy()

    numerical_cols = [
        col for col in feature_groups["numerical"]
        if col in feature_df.columns
    ]

    if numerical_cols:

        scaler = StandardScaler()

        feature_df[numerical_cols] = scaler.fit_transform(
            feature_df[numerical_cols]
        )

        edge_scalers[edge_type] = scaler

    scaled_edge_features[edge_type] = feature_df

    print(f"\n{edge_type.upper()}")
    print("-" * 45)
    print("Feature shape:", feature_df.shape)
    print("Scaled numerical columns:", numerical_cols)

print("\nRelationship numerical scaling completed successfully!")



SUPPLIES_TO
---------------------------------------------
Feature shape: (25, 8)
Scaled numerical columns: ['lead_time_days', 'volume', 'dependency_weight', 'n_skus']

SHIPS_VIA
---------------------------------------------
Feature shape: (9, 4)
Scaled numerical columns: ['lead_time_days', 'volume', 'dependency_weight', 'distance_km']

ROUTE_TO
---------------------------------------------
Feature shape: (50, 15)
Scaled numerical columns: ['lead_time_days', 'volume', 'distance_km', 'avg_freight_cost_usd', 'avg_port_congestion', 'dependency_weight']

DELIVERS_TO
---------------------------------------------
Feature shape: (12, 3)
Scaled numerical columns: ['lead_time_days', 'volume', 'dependency_weight']

HAD_DISRUPTION
---------------------------------------------
Feature shape: (12, 0)
Scaled numerical columns: []

Relationship numerical scaling completed successfully!


In [25]:

# Validate processed relationship features and graph connectivity.

print("PROCESSED RELATIONSHIP FEATURE VALIDATION")
print("=" * 65)

for edge_type, feature_df in scaled_edge_features.items():

    original_count = len(edges[edge_type])
    processed_count = len(feature_df)
    connectivity_count = len(edge_index_frames[edge_type])

    print(f"\n{edge_type.upper()}")
    print("-" * 50)

    print("Original relationships:", original_count)
    print("Processed rows:", processed_count)
    print("Feature columns:", feature_df.shape[1])
    print("Connectivity rows:", connectivity_count)

    print(
        "All numerical:",
        all(
            pd.api.types.is_numeric_dtype(dtype)
            for dtype in feature_df.dtypes
        )
    )

    print("Missing values:", feature_df.isnull().sum().sum())

    infinite_count = (
        np.isinf(feature_df.to_numpy(dtype=float)).sum()
        if feature_df.shape[1] > 0 else 0
    )

    print("Infinite values:", infinite_count)

    assert original_count == processed_count
    assert processed_count == connectivity_count
    assert feature_df.isnull().sum().sum() == 0
    assert infinite_count == 0
    assert all(
        pd.api.types.is_numeric_dtype(dtype)
        for dtype in feature_df.dtypes
    )

print("\nAll relationship feature validation checks passed!")


PROCESSED RELATIONSHIP FEATURE VALIDATION

SUPPLIES_TO
--------------------------------------------------
Original relationships: 25
Processed rows: 25
Feature columns: 8
Connectivity rows: 25
All numerical: True
Missing values: 0
Infinite values: 0

SHIPS_VIA
--------------------------------------------------
Original relationships: 9
Processed rows: 9
Feature columns: 4
Connectivity rows: 9
All numerical: True
Missing values: 0
Infinite values: 0

ROUTE_TO
--------------------------------------------------
Original relationships: 50
Processed rows: 50
Feature columns: 15
Connectivity rows: 50
All numerical: True
Missing values: 0
Infinite values: 0

DELIVERS_TO
--------------------------------------------------
Original relationships: 12
Processed rows: 12
Feature columns: 3
Connectivity rows: 12
All numerical: True
Missing values: 0
Infinite values: 0

HAD_DISRUPTION
--------------------------------------------------
Original relationships: 12
Processed rows: 12
Feature columns: 0
C

In [26]:

# Inspect node identifiers and relationship endpoints before graph construction.

print("NODE DATASET COLUMNS")
print("=" * 65)

for node_type, df in nodes.items():

    print(f"\n{node_type.upper()}")
    print("Columns:", df.columns.tolist())
    print("First 3 records:")
    display(df.head(3))

print("\nRELATIONSHIP ENDPOINTS")
print("=" * 65)

for edge_type, df in edge_index_frames.items():

    print(f"\n{edge_type.upper()}")
    print("First 5 connections:")
    display(df.head())


NODE DATASET COLUMNS

SUPPLIERS
Columns: ['id', 'name', 'country', 'country_iso3', 'lat', 'lon', 'primary_city', 'sku_count', 'avg_defect_rate', 'avg_supplier_lead_time_days', 'inspection_fail_share', 'geo_source', 'source']
First 3 records:


,id,name,country,country_iso3,lat,lon,primary_city,sku_count,avg_defect_rate,avg_supplier_lead_time_days,inspection_fail_share,geo_source,source
0,SUP_001,Supplier 1,India,IND,22.5726,88.3639,Kolkata,27,1.804,14.78,0.222,derived_modal_city,derived
1,SUP_002,Supplier 2,India,IND,28.6139,77.2090,Delhi,22,2.363,18.55,0.364,derived_modal_city,derived
2,SUP_003,Supplier 3,India,IND,13.0827,80.2707,Chennai,15,2.466,20.13,0.200,derived_modal_city,derived



MANUFACTURERS
Columns: ['id', 'name', 'country', 'country_iso3', 'lat', 'lon', 'city', 'sku_count', 'production_volume', 'avg_manufacturing_lead_time_days', 'avg_manufacturing_cost', 'avg_defect_rate', 'source']
First 3 records:


,id,name,country,country_iso3,lat,lon,city,sku_count,production_volume,avg_manufacturing_lead_time_days,avg_manufacturing_cost,avg_defect_rate,source
0,MFG_MUM,Mumbai Manufacturing Hub,India,IND,19.0760,72.8777,Mumbai,22,13160,19.73,36.73,2.122,derived
1,MFG_DEL,Delhi Manufacturing Hub,India,IND,28.6139,77.2090,Delhi,15,8362,13.53,48.88,2.229,derived
2,MFG_BLR,Bangalore Manufacturing Hub,India,IND,12.9716,77.5946,Bangalore,18,7827,11.78,61.99,2.094,derived



PORTS
Columns: ['id', 'name', 'unlocode', 'country', 'country_iso3', 'lat', 'lon', 'hist_disruption_events', 'hist_avg_affected_days', 'hist_max_severity', 'geo_source', 'source']
First 3 records:


,id,name,unlocode,country,country_iso3,lat,lon,hist_disruption_events,hist_avg_affected_days,hist_max_severity,geo_source,source
0,PORT_INNSA,Nhava Sheva (JNPT),INNSA,India,IND,18.95,72.95,0,NaN,NaN,approximate_manual,reference
1,PORT_INMUN,Mundra,INMUN,India,IND,22.74,69.70,0,NaN,NaN,approximate_manual,reference
2,PORT_INMAA,Chennai,INMAA,India,IND,13.09,80.29,0,NaN,NaN,approximate_manual,reference


... (10 outputs left out)


DELIVERS_TO
First 5 connections:


,start_id,end_id
0,PORT_CNSHA,DIST_CHN_01
1,PORT_USNYC,DIST_USA_01
2,PORT_JPTYO,DIST_JPN_01
3,PORT_DEHAM,DIST_DEU_01
4,PORT_GBFXT,DIST_GBR_01



HAD_DISRUPTION
First 5 connections:


,start_id,end_id
0,PORT_CNSHA,EVT_001
1,PORT_USNYC,EVT_002
2,PORT_USNYC,EVT_003
3,PORT_USNYC,EVT_004
4,PORT_USNYC,EVT_005


In [28]:

# Create a global numerical index mapping for all graph nodes.

node_id_to_index = {}
node_type_by_id = {}
node_index_maps = {}

current_index = 0

for node_type, df in nodes.items():

    node_index_maps[node_type] = {}

    for node_id in df["id"]:

        if node_id in node_id_to_index:
            raise ValueError(f"Duplicate node ID found: {node_id}")

        node_id_to_index[node_id] = current_index
        node_index_maps[node_type][node_id] = current_index
        node_type_by_id[node_id] = node_type

        current_index += 1

print("NODE ID MAPPING")
print("=" * 55)

print("Total mapped nodes:", len(node_id_to_index))

for node_type, mapping in node_index_maps.items():

    print(f"\n{node_type.upper()}")
    print("Mapped nodes:", len(mapping))
    print("Sample mapping:", list(mapping.items())[:3])

print("\nGlobal node ID mapping created successfully!")


NODE ID MAPPING
Total mapped nodes: 45

SUPPLIERS
Mapped nodes: 5
Sample mapping: [('SUP_001', 0), ('SUP_002', 1), ('SUP_003', 2)]

MANUFACTURERS
Mapped nodes: 5
Sample mapping: [('MFG_MUM', 5), ('MFG_DEL', 6), ('MFG_BLR', 7)]

PORTS
Mapped nodes: 13
Sample mapping: [('PORT_INNSA', 10), ('PORT_INMUN', 11), ('PORT_INMAA', 12)]

DISTRIBUTORS
Mapped nodes: 10
Sample mapping: [('DIST_CHN_01', 23), ('DIST_USA_01', 24), ('DIST_JPN_01', 25)]

DISRUPTION_EVENTS
Mapped nodes: 12
Sample mapping: [('EVT_001', 33), ('EVT_002', 34), ('EVT_003', 35)]

Global node ID mapping created successfully!


In [29]:

# Validate relationship endpoints against the global node ID mapping.

expected_edge_types = {
    "supplies_to": ("suppliers", "manufacturers"),
    "ships_via": ("manufacturers", "ports"),
    "route_to": ("ports", "ports"),
    "delivers_to": ("ports", "distributors"),
    "had_disruption": ("ports", "disruption_events")
}

print("GRAPH CONNECTIVITY VALIDATION")
print("=" * 65)

total_valid_edges = 0

for edge_type, df in edge_index_frames.items():

    source_type, destination_type = expected_edge_types[edge_type]

    source_ids = df["start_id"]
    destination_ids = df["end_id"]

    missing_sources = [
        node_id for node_id in source_ids
        if node_id not in node_id_to_index
    ]

    missing_destinations = [
        node_id for node_id in destination_ids
        if node_id not in node_id_to_index
    ]

    wrong_source_types = [
        node_id for node_id in source_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != source_type
    ]

    wrong_destination_types = [
        node_id for node_id in destination_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != destination_type
    ]

    valid = (
        not missing_sources
        and not missing_destinations
        and not wrong_source_types
        and not wrong_destination_types
    )

    print(f"\n{edge_type.upper()}")
    print("Relationship count:", len(df))
    print("Missing source IDs:", len(missing_sources))
    print("Missing destination IDs:", len(missing_destinations))
    print("Incorrect source types:", len(wrong_source_types))
    print("Incorrect destination types:", len(wrong_destination_types))
    print("Validation:", "PASSED" if valid else "FAILED")

    assert valid, f"Connectivity validation failed for {edge_type}"

    total_valid_edges += len(df)

print("\nTotal valid relationships:", total_valid_edges)
assert total_valid_edges == 108

print("All graph connectivity checks passed!")


GRAPH CONNECTIVITY VALIDATION

SUPPLIES_TO
Relationship count: 25
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

SHIPS_VIA
Relationship count: 9
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

ROUTE_TO
Relationship count: 50
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

DELIVERS_TO
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

HAD_DISRUPTION
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

Total valid relationships: 108
All graph connectivity checks passed!


In [30]:

# Validate relationship endpoints against the global node ID mapping.

expected_edge_types = {
    "supplies_to": ("suppliers", "manufacturers"),
    "ships_via": ("manufacturers", "ports"),
    "route_to": ("ports", "ports"),
    "delivers_to": ("ports", "distributors"),
    "had_disruption": ("ports", "disruption_events")
}

print("GRAPH CONNECTIVITY VALIDATION")
print("=" * 65)

total_valid_edges = 0

for edge_type, df in edge_index_frames.items():

    source_type, destination_type = expected_edge_types[edge_type]

    source_ids = df["start_id"]
    destination_ids = df["end_id"]

    missing_sources = [
        node_id for node_id in source_ids
        if node_id not in node_id_to_index
    ]

    missing_destinations = [
        node_id for node_id in destination_ids
        if node_id not in node_id_to_index
    ]

    wrong_source_types = [
        node_id for node_id in source_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != source_type
    ]

    wrong_destination_types = [
        node_id for node_id in destination_ids
        if node_id in node_type_by_id
        and node_type_by_id[node_id] != destination_type
    ]

    valid = (
        not missing_sources
        and not missing_destinations
        and not wrong_source_types
        and not wrong_destination_types
    )

    print(f"\n{edge_type.upper()}")
    print("Relationship count:", len(df))
    print("Missing source IDs:", len(missing_sources))
    print("Missing destination IDs:", len(missing_destinations))
    print("Incorrect source types:", len(wrong_source_types))
    print("Incorrect destination types:", len(wrong_destination_types))
    print("Validation:", "PASSED" if valid else "FAILED")

    assert valid, f"Connectivity validation failed for {edge_type}"

    total_valid_edges += len(df)

print("\nTotal valid relationships:", total_valid_edges)
assert total_valid_edges == 108

print("All graph connectivity checks passed!")


GRAPH CONNECTIVITY VALIDATION

SUPPLIES_TO
Relationship count: 25
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

SHIPS_VIA
Relationship count: 9
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

ROUTE_TO
Relationship count: 50
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

DELIVERS_TO
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

HAD_DISRUPTION
Relationship count: 12
Missing source IDs: 0
Missing destination IDs: 0
Incorrect source types: 0
Incorrect destination types: 0
Validation: PASSED

Total valid relationships: 108
All graph connectivity checks passed!


In [31]:

# Convert graph relationships into numerical edge index arrays.

edge_index_arrays = {}

for edge_type, df in edge_index_frames.items():

    source_indices = df["start_id"].map(node_id_to_index).to_numpy()
    destination_indices = df["end_id"].map(node_id_to_index).to_numpy()

    edge_index = np.vstack([
        source_indices,
        destination_indices
    ])

    edge_index_arrays[edge_type] = edge_index

    print(f"{edge_type.upper()}")
    print("Edge index shape:", edge_index.shape)
    print("First 3 edges:")
    print(edge_index[:, :3])
    print("-" * 50)

print("All relationship types converted into numerical edge indices!")


SUPPLIES_TO
Edge index shape: (2, 25)
First 3 edges:
[[0 0 0]
 [7 8 6]]
--------------------------------------------------
SHIPS_VIA
Edge index shape: (2, 9)
First 3 edges:
[[ 5  5  6]
 [10 11 11]]
--------------------------------------------------
ROUTE_TO
Edge index shape: (2, 50)
First 3 edges:
[[10 16 22]
 [15 14 14]]
--------------------------------------------------
DELIVERS_TO
Edge index shape: (2, 12)
First 3 edges:
[[14 15 16]
 [23 24 25]]
--------------------------------------------------
HAD_DISRUPTION
Edge index shape: (2, 12)
First 3 edges:
[[14 15 15]
 [33 34 35]]
--------------------------------------------------
All relationship types converted into numerical edge indices!


In [32]:

# Convert processed node and relationship features into numerical matrices.

node_feature_arrays = {}
edge_feature_arrays = {}

print("NODE FEATURE MATRICES")
print("=" * 55)

for node_type, df in scaled_node_features.items():

    feature_array = df.to_numpy(dtype=np.float32)

    node_feature_arrays[node_type] = feature_array

    print(f"{node_type.upper()}:")
    print("Feature matrix shape:", feature_array.shape)
    print("Data type:", feature_array.dtype)

print("\nEDGE FEATURE MATRICES")
print("=" * 55)

for edge_type, df in scaled_edge_features.items():

    feature_array = df.to_numpy(dtype=np.float32)

    edge_feature_arrays[edge_type] = feature_array

    print(f"{edge_type.upper()}:")
    print("Feature matrix shape:", feature_array.shape)
    print("Data type:", feature_array.dtype)

print("\nFeature matrices created successfully!")


NODE FEATURE MATRICES
SUPPLIERS:
Feature matrix shape: (5, 9)
Data type: float32
MANUFACTURERS:
Feature matrix shape: (5, 12)
Data type: float32
PORTS:
Feature matrix shape: (13, 17)
Data type: float32
DISTRIBUTORS:
Feature matrix shape: (10, 12)
Data type: float32
DISRUPTION_EVENTS:
Feature matrix shape: (12, 1)
Data type: float32

EDGE FEATURE MATRICES
SUPPLIES_TO:
Feature matrix shape: (25, 8)
Data type: float32
SHIPS_VIA:
Feature matrix shape: (9, 4)
Data type: float32
ROUTE_TO:
Feature matrix shape: (50, 15)
Data type: float32
DELIVERS_TO:
Feature matrix shape: (12, 3)
Data type: float32
HAD_DISRUPTION:
Feature matrix shape: (12, 0)
Data type: float32

Feature matrices created successfully!
